<a href="https://colab.research.google.com/github/HiteshTirumalasetti/ExpenseManagemetApp/blob/main/Car_Damage_EfficientDet_Colab(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Car Damage Detection with EfficientDet-D0

**Dataset:** [CarDD - Car Damage Detection](https://www.kaggle.com/datasets/nasimetemadi/car-damage-detection)  
**Model:** EfficientDet-D0 (pretrained on COCO, fine-tuned on 6 damage classes)  
**Classes:** dent, scratch, crack, glass shatter, lamp broken, tire flat  
**Runtime:** ~1.5-2 hours on free T4 GPU

---
### How to run:
1. Click **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. When prompted for Kaggle credentials, paste your `kaggle.json` contents
4. Sit back and wait — results appear at the bottom

### How to get kaggle.json:
1. Go to https://www.kaggle.com/settings
2. Scroll to **API** section
3. Click **Create New Token** — downloads `kaggle.json`
4. You'll paste the contents when prompted below

## Cell 1: Verify GPU & Install Dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

!pip install -q effdet timm albumentations kagglehub

## Cell 2: Setup Kaggle & Download Dataset

In [ ]:
import os
from pathlib import Path

# ---- Kaggle credentials ----
# Option A: Upload kaggle.json manually
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
if not kaggle_json.exists():
    print("Kaggle credentials not found. Setting up...")
    print("Paste the CONTENTS of your kaggle.json below (looks like: {\"username\":\"xxx\",\"key\":\"xxx\"})")
    print("Get it from: https://www.kaggle.com/settings -> API -> Create New Token\n")
    creds = input("Paste kaggle.json contents: ").strip()
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_text(creds)
    os.chmod(str(kaggle_json), 0o600)
    print("Kaggle credentials saved!")
else:
    print("Kaggle credentials found.")

# ---- Download dataset ----
import kagglehub
print("\nDownloading CarDD dataset (5.6 GB, may take a few minutes)...")
dataset_path = kagglehub.dataset_download("nasimetemadi/car-damage-detection")
print(f"Dataset at: {dataset_path}")

# Find the COCO directory
dataset_root = Path(dataset_path)
coco_candidates = list(dataset_root.rglob("CarDD_COCO"))
if coco_candidates:
    COCO_DIR = coco_candidates[0]
else:
    ann_candidates = list(dataset_root.rglob("instances_train2017.json"))
    COCO_DIR = ann_candidates[0].parent.parent

PATHS = {
    "train_images": str(COCO_DIR / "train2017"),
    "val_images": str(COCO_DIR / "val2017"),
    "test_images": str(COCO_DIR / "test2017"),
    "train_ann": str(COCO_DIR / "annotations" / "instances_train2017.json"),
    "val_ann": str(COCO_DIR / "annotations" / "instances_val2017.json"),
    "test_ann": str(COCO_DIR / "annotations" / "instances_test2017.json"),
}

for k, v in PATHS.items():
    exists = os.path.exists(v)
    count = len(os.listdir(v)) if exists and os.path.isdir(v) else "-"
    print(f"  {k}: {v}  [{'OK' if exists else 'MISSING'}] {count} files" if os.path.isdir(v) and exists else f"  {k}: {'OK' if exists else 'MISSING'}")

## Cell 3: Configuration

In [ ]:
# =============================================================================
# CONFIGURATION — edit these if you want
# =============================================================================
BATCH_SIZE = 8              # T4 can handle 8 at 512x512
LEARNING_RATE_BACKBONE = 1e-4
LEARNING_RATE_HEAD = 5e-4
NUM_EPOCHS = 30             # Usually converges in 15-25 epochs
SUBSET_SIZE = None          # None = use ALL images (2816 train)
CONFIDENCE_THRESHOLD = 0.3
IOU_THRESHOLD = 0.5
EARLY_STOPPING_PATIENCE = 7
RANDOM_SEED = 42
IMAGE_SIZE = 512            # EfficientDet-D0 native size
NUM_WORKERS = 2             # Colab can handle 2
GRAD_ACCUMULATION_STEPS = 2 # Effective batch = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE} x {GRAD_ACCUMULATION_STEPS} accumulation = {BATCH_SIZE * GRAD_ACCUMULATION_STEPS} effective")
print(f"Training for up to {NUM_EPOCHS} epochs with patience={EARLY_STOPPING_PATIENCE}")

## Cell 4: Load COCO Annotations

In [ ]:
import json
from collections import defaultdict

def load_coco_annotations(ann_path):
    """Load COCO annotations → image info, annotations grouped by image, categories."""
    with open(ann_path, "r") as f:
        coco = json.load(f)

    categories = {cat["id"]: cat["name"] for cat in coco["categories"]}
    images_info = {
        img["id"]: {"file_name": img["file_name"], "width": img["width"], "height": img["height"]}
        for img in coco["images"]
    }
    annotations = defaultdict(list)
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        annotations[ann["image_id"]].append({
            "bbox": [x, y, x + w, y + h],
            "category_id": ann["category_id"],
        })

    print(f"  {ann_path.split('/')[-1]}: {len(images_info)} images, "
          f"{sum(len(v) for v in annotations.values())} annotations")
    return images_info, dict(annotations), categories

print("Loading annotations...")
train_imgs, train_anns, categories = load_coco_annotations(PATHS["train_ann"])
val_imgs, val_anns, _ = load_coco_annotations(PATHS["val_ann"])
test_imgs, test_anns, _ = load_coco_annotations(PATHS["test_ann"])

id_to_class = dict(categories)
id_to_class[0] = "background"
num_classes = len(categories)

print(f"\n{num_classes} damage classes:")
for cid, name in sorted(categories.items()):
    print(f"  {cid}: {name}")

## Cell 5: Preprocessing & Dataset Class

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


def validate_image_quality(image_np, min_size=50, min_std=5.0):
    """Quality gate: not None, not tiny, not blank."""
    if image_np is None:
        return False, "Load failed"
    h, w = image_np.shape[:2]
    if h < min_size or w < min_size:
        return False, f"Too small: {w}x{h}"
    if image_np.std() < min_std:
        return False, f"Blank (std={image_np.std():.1f})"
    return True, "OK"


def correct_angle(image_np):
    """Correct small skew via Hough lines (within +/-15 degrees)."""
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=100, maxLineGap=10)
    if lines is None:
        return image_np
    angles = [np.degrees(np.arctan2(l[0][3]-l[0][1], l[0][2]-l[0][0])) for l in lines]
    median_angle = np.median(angles)
    if abs(median_angle) > 15:
        return image_np
    h, w = image_np.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), median_angle, 1.0)
    return cv2.warpAffine(image_np, M, (w, h), borderMode=cv2.BORDER_REPLICATE)


def get_train_transforms(image_size):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.GaussNoise(p=0.2),
        A.Resize(height=image_size, width=image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_area=1.0, min_visibility=0.1))


def get_val_transforms(image_size):
    return A.Compose([
        A.Resize(height=image_size, width=image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_area=1.0, min_visibility=0.1))


class CarDamageDataset(Dataset):
    """COCO-format car damage dataset for EfficientDet."""

    def __init__(self, img_dir, images_info, annotations, transforms=None,
                 apply_preprocessing=True, subset_size=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.apply_preprocessing = apply_preprocessing
        self.samples = []
        skipped = 0

        image_ids = sorted(images_info.keys())
        if subset_size and subset_size < len(image_ids):
            rng = np.random.RandomState(RANDOM_SEED)
            image_ids = list(rng.choice(image_ids, size=subset_size, replace=False))

        for img_id in image_ids:
            info = images_info[img_id]
            img_path = os.path.join(img_dir, info["file_name"])
            if not os.path.exists(img_path):
                skipped += 1; continue
            anns = annotations.get(img_id, [])
            if not anns:
                skipped += 1; continue

            w, h = info["width"], info["height"]
            valid_anns = []
            for ann in anns:
                x0, y0, x1, y1 = ann["bbox"]
                x0 = max(0.0, min(x0, w-1))
                y0 = max(0.0, min(y0, h-1))
                x1 = max(x0+1, min(x1, float(w)))
                y1 = max(y0+1, min(y1, float(h)))
                if x1 > x0+1 and y1 > y0+1:
                    valid_anns.append({"bbox": [x0, y0, x1, y1], "category_id": ann["category_id"]})

            if not valid_anns:
                skipped += 1; continue

            self.samples.append({"file_name": info["file_name"], "annotations": valid_anns, "width": w, "height": h})

        print(f"    {len(self.samples)} valid, {skipped} skipped")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image_np = cv2.imread(os.path.join(self.img_dir, sample["file_name"]))
        image_np = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)
        if self.apply_preprocessing:
            image_np = correct_angle(image_np)

        h, w = image_np.shape[:2]
        boxes, labels = [], []
        for ann in sample["annotations"]:
            x0, y0, x1, y1 = ann["bbox"]
            boxes.append([max(0,min(x0,w-1)), max(0,min(y0,h-1)), max(1,min(x1,float(w))), max(1,min(y1,float(h)))])
            labels.append(ann["category_id"])

        if self.transforms:
            t = self.transforms(image=image_np, bboxes=boxes, labels=labels)
            image_tensor = t["image"]
            boxes = t["bboxes"]
            labels = t["labels"]
        else:
            image_tensor = torch.from_numpy(image_np).permute(2,0,1).float() / 255.0

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        return image_tensor, {"boxes": boxes, "labels": labels,
                              "img_scale": torch.tensor([1.0]),
                              "img_size": torch.tensor([IMAGE_SIZE, IMAGE_SIZE])}


def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images), list(targets)


print("Building datasets...")
print("  Train:")
train_dataset = CarDamageDataset(
    PATHS["train_images"], train_imgs, train_anns,
    get_train_transforms(IMAGE_SIZE), subset_size=SUBSET_SIZE)
print("  Val:")
val_dataset = CarDamageDataset(
    PATHS["val_images"], val_imgs, val_anns,
    get_val_transforms(IMAGE_SIZE))
print("  Test:")
test_dataset = CarDamageDataset(
    PATHS["test_images"], test_imgs, test_anns,
    get_val_transforms(IMAGE_SIZE))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nDataloaders ready: {len(train_loader)} train batches, {len(val_loader)} val, {len(test_loader)} test")

## Cell 6: Build EfficientDet Model

In [ ]:
import torch.nn as nn
from effdet import create_model_from_config, get_efficientdet_config
from effdet.anchors import Anchors, AnchorLabeler, generate_detections
from effdet.loss import DetectionLoss
import torch.hub


class EfficientDetModel(nn.Module):
    def __init__(self, num_classes, image_size=512, arch="tf_efficientdet_d0"):
        super().__init__()
        self.num_classes = num_classes
        self.image_size = image_size
        config = get_efficientdet_config(arch)
        config.image_size = [image_size, image_size]
        config.num_classes = num_classes
        config.soft_nms = False
        config.max_det_per_image = 100

        config_pretrained = get_efficientdet_config(arch)
        config_pretrained.image_size = [image_size, image_size]
        self.net = create_model_from_config(
            config_pretrained, bench_task='', pretrained=True, num_classes=num_classes)
        self.config = config
        print(f"  Model created with pretrained COCO backbone")

        self.anchors = Anchors.from_config(config)
        self.anchor_labeler = AnchorLabeler(self.anchors, num_classes, match_threshold=0.5)
        self.loss_fn = DetectionLoss(config)
        self.num_levels = self.anchors.max_level - self.anchors.min_level + 1

    def forward(self, images, targets=None):
        # net returns lists of tensors, one per FPN level
        class_out, box_out = self.net(images)  # each is a list of [batch, anchors*classes, H, W]

        if self.training and targets is not None:
            bs = images.shape[0]

            # label_anchors returns (cls_list, box_list, num_pos)
            # where cls_list/box_list are lists of tensors per FPN level
            all_cls = []  # will be list of lists
            all_box = []
            all_num_pos = []

            for i in range(bs):
                gt_boxes = targets[i]["boxes"].to(images.device).float()
                gt_labels = targets[i]["labels"].to(images.device).long()

                if gt_boxes.shape[0] == 0:
                    cls_t, box_t, num_pos = self.anchor_labeler.label_anchors(
                        torch.zeros(0, 4, device=images.device),
                        torch.zeros(0, dtype=torch.long, device=images.device))
                else:
                    # Convert [x1,y1,x2,y2] to [y1,x1,y2,x2]
                    gt_boxes_yxyx = torch.stack([
                        gt_boxes[:,1], gt_boxes[:,0], gt_boxes[:,3], gt_boxes[:,2]
                    ], dim=1)
                    cls_t, box_t, num_pos = self.anchor_labeler.label_anchors(
                        gt_boxes_yxyx, gt_labels)

                all_cls.append(cls_t)   # cls_t is a list of tensors per level
                all_box.append(box_t)   # box_t is a list of tensors per level
                all_num_pos.append(num_pos)

            # Stack per-level: loss_fn expects List[Tensor] where each tensor is [batch, ...]
            cls_targets = []
            box_targets = []
            for level_idx in range(self.num_levels):
                cls_targets.append(torch.stack([all_cls[i][level_idx] for i in range(bs)]))
                box_targets.append(torch.stack([all_box[i][level_idx] for i in range(bs)]))

            num_positives = sum(all_num_pos)
            if isinstance(num_positives, int):
                num_positives = torch.tensor(num_positives, dtype=torch.float32, device=images.device)

            loss, class_loss, box_loss = self.loss_fn(
                class_out, box_out, cls_targets, box_targets, num_positives)
            return {"loss": loss, "class_loss": class_loss, "box_loss": box_loss}
        else:
            return class_out, box_out

    def predict(self, images):
        self.eval()
        with torch.no_grad():
            class_out, box_out = self.net(images)
        results = []
        for i in range(images.shape[0]):
            cls_i = [c[i:i+1] for c in class_out]
            box_i = [b[i:i+1] for b in box_out]
            detections = generate_detections(
                cls_i, box_i, self.anchors.boxes,
                indices=None, classes=None,
                img_scale=torch.tensor([1.0], device=images.device),
                img_size=torch.tensor([[self.image_size, self.image_size]], device=images.device),
                max_det_per_image=self.config.max_det_per_image,
                soft_nms=self.config.soft_nms)
            dets = detections[0]
            mask = dets[:, 4] > 0.01
            dets = dets[mask]
            results.append({"boxes": dets[:,:4], "scores": dets[:,4], "labels": dets[:,5].int()+1})
        return results


print(f"Building EfficientDet-D0 ({num_classes} classes)...")
model = EfficientDetModel(num_classes=num_classes, image_size=IMAGE_SIZE)
model.to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params:     {total_p:,}")
print(f"  Trainable params: {train_p:,}")

backbone_params, head_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad: continue
    (backbone_params if "backbone" in name or "fpn" in name else head_params).append(param)

optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LEARNING_RATE_BACKBONE},
    {"params": head_params, "lr": LEARNING_RATE_HEAD},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)

print(f"  Backbone+FPN: {sum(p.numel() for p in backbone_params):,} params (lr={LEARNING_RATE_BACKBONE})")
print(f"  Head:         {sum(p.numel() for p in head_params):,} params (lr={LEARNING_RATE_HEAD})")
print("  Model ready!")

## Cell 7: Training Loop (~1-1.5 hours on T4)

In [ ]:
from tqdm.notebook import tqdm
import time

def train_one_epoch(model, dataloader, optimizer, device, accumulation_steps):
    model.train()
    total_loss, total_cls, total_box, n = 0, 0, 0, 0
    optimizer.zero_grad()
    for batch_idx, (images, targets) in enumerate(tqdm(dataloader, desc="  Train", leave=False)):
        images = images.to(device)
        try:
            output = model(images, targets)
        except Exception as e:
            print(f"  [WARN] Batch {batch_idx} error: {e}")
            continue
        loss = output["loss"] / accumulation_steps
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        loss.backward()
        if (batch_idx+1) % accumulation_steps == 0 or (batch_idx+1) == len(dataloader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        total_loss += output["loss"].item()
        total_cls += output["class_loss"].item()
        total_box += output["box_loss"].item()
        n += 1
    return {"loss": total_loss/max(n,1), "cls": total_cls/max(n,1), "box": total_box/max(n,1)}

def validate_one_epoch(model, dataloader, device):
    model.train()
    total_loss, n = 0, 0
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="  Val", leave=False):
            images = images.to(device)
            try:
                output = model(images, targets)
                if not (torch.isnan(output["loss"]) or torch.isinf(output["loss"])):
                    total_loss += output["loss"].item()
                    n += 1
            except: continue
    return total_loss / max(n, 1)

print(f"Training for up to {NUM_EPOCHS} epochs (patience={EARLY_STOPPING_PATIENCE})")
print(f"Estimated time: ~1-1.5 hours on T4 GPU\n")
best_val_loss = float("inf")
epochs_no_improve = 0
save_path = "/content/efficientdet_car_damage_best.pt"
history = {"train_loss": [], "val_loss": []}
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    train_losses = train_one_epoch(model, train_loader, optimizer, DEVICE, GRAD_ACCUMULATION_STEPS)
    scheduler.step()
    val_loss = validate_one_epoch(model, val_loader, DEVICE)
    history["train_loss"].append(train_losses["loss"])
    history["val_loss"].append(val_loss)
    elapsed = time.time() - epoch_start
    total_elapsed = time.time() - start_time
    print(f"  Train: {train_losses['loss']:.4f} (cls:{train_losses['cls']:.4f} box:{train_losses['box']:.4f})")
    print(f"  Val:   {val_loss:.4f}  [{elapsed:.0f}s, total {total_elapsed/60:.1f}min]")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), save_path)
        print(f"  >> Saved best model (val_loss={best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{EARLY_STOPPING_PATIENCE})")
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}!")
            break
    print()

total_time = time.time() - start_time
print(f"\nTraining complete! Total time: {total_time/60:.1f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")


## Cell 8: Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train Loss", marker="o", markersize=4)
plt.plot(history["val_loss"], label="Val Loss", marker="s", markersize=4)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientDet-D0 Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 9: Evaluate on Test Set

In [ ]:
save_path = "/content/efficientdet_car_damage_best.pt"
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Load best model
print("Loading best model...")
model.load_state_dict(torch.load(save_path, map_location=DEVICE, weights_only=True))
model.eval()


def compute_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    union = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter
    return inter / max(union, 1e-6)


print(f"Evaluating on {len(test_dataset)} test images...")
all_pred, all_true = [], []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="  Testing"):
        images = images.to(DEVICE)
        try:
            results = model.predict(images)
        except Exception as e:
            continue

        for i, target in enumerate(targets):
            gt_boxes = target["boxes"].cpu().numpy()
            gt_labels = target["labels"].cpu().numpy()
            pred_boxes = results[i]["boxes"].cpu().numpy()
            pred_scores = results[i]["scores"].cpu().numpy()
            pred_labels = results[i]["labels"].cpu().numpy()

            mask = pred_scores >= CONFIDENCE_THRESHOLD
            pred_boxes, pred_scores, pred_labels = pred_boxes[mask], pred_scores[mask], pred_labels[mask]

            matched_gt = set()
            for pi in np.argsort(-pred_scores):
                best_iou, best_gi = 0, -1
                for gi in range(len(gt_boxes)):
                    if gi in matched_gt: continue
                    iou = compute_iou(pred_boxes[pi], gt_boxes[gi])
                    if iou > best_iou: best_iou, best_gi = iou, gi
                if best_iou >= IOU_THRESHOLD and best_gi >= 0:
                    matched_gt.add(best_gi)
                    all_pred.append(int(pred_labels[pi]))
                    all_true.append(int(gt_labels[best_gi]))
                else:
                    all_pred.append(int(pred_labels[pi]))
                    all_true.append(0)
            for gi in range(len(gt_boxes)):
                if gi not in matched_gt:
                    all_pred.append(0)
                    all_true.append(int(gt_labels[gi]))

all_pred_names = [id_to_class.get(l, "background") for l in all_pred]
all_true_names = [id_to_class.get(l, "background") for l in all_true]
print("Evaluation complete!")

In [ ]:
# Fix predict + re-evaluate in one cell
from effdet.anchors import decode_box_outputs
import torch

def predict_images(model, images, conf_thresh=0.3):
    """Simple predict without generate_detections."""
    model.eval()
    with torch.no_grad():
        class_out, box_out = model.net(images)

    # Flatten all FPN levels into single tensors
    cls_list, box_list = [], []
    for cls_level, box_level in zip(class_out, box_out):
        bs, c, h, w = cls_level.shape
        cls_list.append(cls_level.permute(0, 2, 3, 1).reshape(bs, -1, model.num_classes))
        box_list.append(box_level.permute(0, 2, 3, 1).reshape(bs, -1, 4))

    cls_preds = torch.cat(cls_list, dim=1).sigmoid()  # [bs, total_anchors, num_classes]
    box_preds = torch.cat(box_list, dim=1)             # [bs, total_anchors, 4]

    # Decode boxes using anchors
    anchor_boxes = model.anchors.boxes.to(images.device)  # [total_anchors, 4]

    results = []
    for i in range(images.shape[0]):
        # Get max class score and label per anchor
        scores_per_class, labels = cls_preds[i].max(dim=1)  # [total_anchors]

        # Filter by confidence
        mask = scores_per_class > conf_thresh
        if mask.sum() == 0:
            results.append({"boxes": torch.zeros(0,4), "scores": torch.zeros(0), "labels": torch.zeros(0, dtype=torch.int)})
            continue

        scores = scores_per_class[mask]
        labels_filt = labels[mask] + 1  # 0-indexed → 1-indexed
        box_filt = box_preds[i][mask]
        anchor_filt = anchor_boxes[mask]

        # Decode: convert deltas to absolute boxes
        # anchors are [y1, x1, y2, x2], deltas are [dy, dx, dh, dw]
        a_y1, a_x1, a_y2, a_x2 = anchor_filt[:, 0], anchor_filt[:, 1], anchor_filt[:, 2], anchor_filt[:, 3]
        a_h = a_y2 - a_y1
        a_w = a_x2 - a_x1
        a_cy = a_y1 + 0.5 * a_h
        a_cx = a_x1 + 0.5 * a_w

        dy, dx, dh, dw = box_filt[:, 0], box_filt[:, 1], box_filt[:, 2], box_filt[:, 3]
        cy = dy * a_h + a_cy
        cx = dx * a_w + a_cx
        h = torch.exp(dh) * a_h
        w = torch.exp(dw) * a_w

        x1 = cx - 0.5 * w
        y1 = cy - 0.5 * h
        x2 = cx + 0.5 * w
        y2 = cy + 0.5 * h

        decoded_boxes = torch.stack([x1, y1, x2, y2], dim=1)
        decoded_boxes = decoded_boxes.clamp(min=0, max=model.image_size)

        # Simple NMS per class
        keep_boxes, keep_scores, keep_labels = [], [], []
        for cls_id in labels_filt.unique():
            cls_mask = labels_filt == cls_id
            c_boxes = decoded_boxes[cls_mask]
            c_scores = scores[cls_mask]
            nms_idx = torch.ops.torchvision.nms(c_boxes, c_scores, 0.5)
            keep_boxes.append(c_boxes[nms_idx])
            keep_scores.append(c_scores[nms_idx])
            keep_labels.append(labels_filt[cls_mask][nms_idx])

        results.append({
            "boxes": torch.cat(keep_boxes) if keep_boxes else torch.zeros(0,4),
            "scores": torch.cat(keep_scores) if keep_scores else torch.zeros(0),
            "labels": torch.cat(keep_labels).int() if keep_labels else torch.zeros(0, dtype=torch.int),
        })
    return results

# ---- Re-run evaluation with fixed predict ----
print("Re-evaluating with fixed predict...")
all_pred, all_true = [], []
model.eval()

from tqdm.notebook import tqdm
for images, targets in tqdm(test_loader, desc="Testing"):
    images = images.to(DEVICE)
    results = predict_images(model, images, conf_thresh=CONFIDENCE_THRESHOLD)

    for i, target in enumerate(targets):
        gt_boxes = target["boxes"].cpu().numpy()
        gt_labels = target["labels"].cpu().numpy()
        pred_boxes = results[i]["boxes"].cpu().numpy()
        pred_scores = results[i]["scores"].cpu().numpy()
        pred_labels = results[i]["labels"].cpu().numpy()

        matched_gt = set()
        for pi in np.argsort(-pred_scores):
            best_iou, best_gi = 0, -1
            for gi in range(len(gt_boxes)):
                if gi in matched_gt: continue
                iou = compute_iou(pred_boxes[pi], gt_boxes[gi])
                if iou > best_iou: best_iou, best_gi = iou, gi
            if best_iou >= IOU_THRESHOLD and best_gi >= 0:
                matched_gt.add(best_gi)
                all_pred.append(int(pred_labels[pi]))
                all_true.append(int(gt_labels[best_gi]))
            else:
                all_pred.append(int(pred_labels[pi]))
                all_true.append(0)
        for gi in range(len(gt_boxes)):
            if gi not in matched_gt:
                all_pred.append(0)
                all_true.append(int(gt_labels[gi]))

all_pred_names = [id_to_class.get(l, "background") for l in all_pred]
all_true_names = [id_to_class.get(l, "background") for l in all_true]

# Print results
tp = sum(1 for t, p in zip(all_true_names, all_pred_names) if t == p and t != "background")
fp = sum(1 for t, p in zip(all_true_names, all_pred_names) if t == "background" and p != "background")
fn = sum(1 for t, p in zip(all_true_names, all_pred_names) if t != "background" and p == "background")
mis = sum(1 for t, p in zip(all_true_names, all_pred_names) if t != p and t != "background" and p != "background")

print("\n" + "=" * 50)
print("CAR DAMAGE DETECTION — FINAL RESULTS")
print("=" * 50)
print(f"  True Positives:    {tp}")
print(f"  False Positives:   {fp}")
print(f"  False Negatives:   {fn}")
print(f"  Misclassified:     {mis}")
print(f"  Total GT objects:  {tp + fn + mis}")
if tp + fp > 0:
    print(f"  Precision:         {tp/(tp+fp):.4f}")
if tp + fn + mis > 0:
    print(f"  Recall:            {tp/(tp+fn+mis):.4f}")
if tp + fp > 0 and tp + fn + mis > 0:
    prec = tp/(tp+fp)
    rec = tp/(tp+fn+mis)
    print(f"  F1 Score:          {2*prec*rec/(prec+rec):.4f}" if prec+rec > 0 else "  F1 Score: 0")
print(f"  Total predictions: {tp + fp + mis}")


## Cell 10: Results — Overall & Per-Class Metrics

In [ ]:
# Quick results — no heavy computation
tp = sum(1 for t, p in zip(all_true_names, all_pred_names) if t == p and t != "background")
fp = sum(1 for t, p in zip(all_true_names, all_pred_names) if t == "background" and p != "background")
fn = sum(1 for t, p in zip(all_true_names, all_pred_names) if t != "background" and p == "background")
mis = sum(1 for t, p in zip(all_true_names, all_pred_names) if t != p and t != "background" and p != "background")
total_gt = tp + fn + mis

print("=" * 50)
print("CAR DAMAGE DETECTION — FINAL RESULTS")
print("=" * 50)
print(f"  True Positives:    {tp}")
print(f"  False Positives:   {fp}")
print(f"  False Negatives:   {fn}")
print(f"  Misclassified:     {mis}")
print(f"  Total GT objects:  {total_gt}")
if tp + fp > 0:
    print(f"  Precision:         {tp/(tp+fp):.4f}")
if tp + fn > 0:
    print(f"  Recall:            {tp/(tp+fn+mis):.4f}")
print(f"  Total predictions: {len(all_pred_names)}")
print(f"  Total ground truth:{len(all_true_names)}")


## Cell 11: Visualize Sample Detections

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

COLORS = {1: "red", 2: "blue", 3: "green", 4: "orange", 5: "purple", 6: "cyan"}
model.eval()
indices = np.random.choice(len(test_dataset), 6, replace=False)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax_idx, data_idx in enumerate(indices):
    img_tensor, target = test_dataset[data_idx]
    images = img_tensor.unsqueeze(0).to(DEVICE)
    results = predict_images(model, images, conf_thresh=CONFIDENCE_THRESHOLD)

    img_np = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img_np = (img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])).clip(0,1)

    ax = axes[ax_idx//3][ax_idx%3]
    ax.imshow(img_np)

    r = results[0]
    for j in range(len(r["scores"])):
        x1,y1,x2,y2 = r["boxes"][j].cpu().numpy()
        lid = r["labels"][j].item()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor=COLORS.get(lid,"white"),facecolor="none"))
        ax.text(x1,y1-5,f"{id_to_class.get(lid,'?')} {r['scores'][j]:.2f}",color=COLORS.get(lid,"white"),fontsize=9,
                bbox=dict(boxstyle="round,pad=0.2",facecolor="black",alpha=0.7))

    for j in range(len(target["boxes"])):
        x1,y1,x2,y2 = target["boxes"][j].numpy()
        lid = target["labels"][j].item()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=1.5,edgecolor="lime",facecolor="none",linestyle="--"))
        ax.text(x2,y2+12,f"GT:{id_to_class.get(lid,'?')}",color="lime",fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2",facecolor="black",alpha=0.5))

    ax.set_title(f"Image {data_idx}")
    ax.axis("off")

plt.suptitle("Predictions (solid) vs Ground Truth (dashed green)", fontsize=14)
plt.tight_layout()
plt.show()


## Cell 12: Download Model to Your Computer

In [ ]:
from google.colab import files

# Save results to file
results_text = []
results_text.append("CAR DAMAGE DETECTION - EfficientDet-D0 Results")
results_text.append(f"Epochs trained: {len(history['train_loss'])}")
results_text.append(f"Best val loss: {best_val_loss:.2f}")
results_text.append(f"Total training time: {total_time/60:.1f} minutes")
results_text.append("")
if filt_true:
    results_text.append(f"Accuracy:  {acc:.4f}")
    results_text.append(f"Precision: {prec:.4f}")
    results_text.append(f"Recall:    {rec:.4f}")
    results_text.append(f"F1 Score:  {f1:.4f}")
    results_text.append("")
    results_text.append(f"TP={tp}  FP={fp}  FN={fn}  Misclass={mis}")

with open("/content/results.txt", "w") as f:
    f.write("\n".join(results_text))

print("Downloading model and results...")
print("(If download doesn't start, click the files icon on the left sidebar)")
files.download("/content/efficientdet_car_damage_best.pt")
files.download("/content/results.txt")